In [1]:
import pandas as pd

df = pd.read_csv('datasets/Titanic-Dataset.csv')

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (891, 12)
Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
df['Age'] = df['Age'].fillna(df['Age'].median())

df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

df['Embarked'] = df['Embarked'].fillna('S')
df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

print("Nulls remaining:")
print(df[['Age', 'Sex', 'Embarked']].isnull().sum())
print("\nSex column after encoding (0=male, 1=female):")
print(df['Sex'].value_counts())

Nulls remaining:
Age         0
Sex         0
Embarked    0
dtype: int64

Sex column after encoding (0=male, 1=female):
Sex
0    577
1    314
Name: count, dtype: int64


In [3]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']

X = df[features]
y = df['Survived']

print("X shape (features):", X.shape)
print("y shape (target):", y.shape)
print("\nFirst 5 rows of X:")
print(X.head())
print("\nFirst 5 values of y (0=died, 1=survived):")
print(y.head())

X shape (features): (891, 7)
y shape (target): (891,)

First 5 rows of X:
   Pclass  Sex   Age  SibSp  Parch     Fare  Embarked
0       3    0  22.0      1      0   7.2500         0
1       1    1  38.0      1      0  71.2833         1
2       3    1  26.0      0      0   7.9250         0
3       1    1  35.0      1      0  53.1000         0
4       3    0  35.0      0      0   8.0500         0

First 5 values of y (0=died, 1=survived):
0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("\nThe model will LEARN from", len(X_train), "passengers")
print("The model will be TESTED on", len(X_test), "passengers it has never seen")

Training rows: 712
Testing rows: 179

The model will LEARN from 712 passengers
The model will be TESTED on 179 passengers it has never seen


In [5]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=5, random_state=42)

model.fit(X_train, y_train)

print("Model trained successfully!")
print("Number of features used:", model.n_features_in_)
print("Feature names:", features)

Model trained successfully!
Number of features used: 7
Feature names: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']


In [6]:
predictions = model.predict(X_test)

print("First 10 predictions (0=died, 1=survived):")
print(predictions[:10])

print("\nActual answers:")
print(y_test.values[:10])

print("\nSide by side comparison:")
for i in range(10):
    pred = predictions[i]
    actual = y_test.values[i]
    result = "CORRECT" if pred == actual else "WRONG"
    print(f"  Predicted: {pred}  |  Actual: {actual}  |  {result}")

First 10 predictions (0=died, 1=survived):
[0 0 0 1 1 1 1 0 1 1]

Actual answers:
[1 0 0 1 1 1 1 0 1 1]

Side by side comparison:
  Predicted: 0  |  Actual: 1  |  WRONG
  Predicted: 0  |  Actual: 0  |  CORRECT
  Predicted: 0  |  Actual: 0  |  CORRECT
  Predicted: 1  |  Actual: 1  |  CORRECT
  Predicted: 1  |  Actual: 1  |  CORRECT
  Predicted: 1  |  Actual: 1  |  CORRECT
  Predicted: 1  |  Actual: 1  |  CORRECT
  Predicted: 0  |  Actual: 0  |  CORRECT
  Predicted: 1  |  Actual: 1  |  CORRECT
  Predicted: 1  |  Actual: 1  |  CORRECT


In [7]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, predictions)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

print("\nDetailed breakdown:")
print(classification_report(y_test, predictions, target_names=['Died', 'Survived']))

Model Accuracy: 79.89%

Detailed breakdown:
              precision    recall  f1-score   support

        Died       0.79      0.90      0.84       105
    Survived       0.83      0.65      0.73        74

    accuracy                           0.80       179
   macro avg       0.81      0.78      0.78       179
weighted avg       0.80      0.80      0.79       179



In [8]:
import pandas as pd

importances = pd.Series(
    model.feature_importances_,
    index=features
).sort_values(ascending=False)

print("Feature importance (higher = more useful for prediction):")
print(importances)

Feature importance (higher = more useful for prediction):
Sex         0.537435
Pclass      0.191388
Fare        0.124937
Age         0.090645
SibSp       0.042707
Embarked    0.012888
Parch       0.000000
dtype: float64


In [9]:
import numpy as np

rich_woman = pd.DataFrame([[1, 1, 29, 0, 0, 100, 0]], columns=features)
poor_man   = pd.DataFrame([[3, 0, 22, 1, 0, 7,   0]], columns=features)

print("Rich woman in 1st class, age 29:")
print("  Prediction:", "SURVIVED" if model.predict(rich_woman)[0] == 1 else "DIED")

print("\nPoor man in 3rd class, age 22:")
print("  Prediction:", "SURVIVED" if model.predict(poor_man)[0] == 1 else "DIED")

Rich woman in 1st class, age 29:
  Prediction: SURVIVED

Poor man in 3rd class, age 22:
  Prediction: DIED
